In [ ]:
!nvidia-smi

Thu Oct 15 21:23:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
print(torch.cuda.is_available())        # True
print(torch.cuda.get_device_name(0))    # Tesla T4

True
Tesla T4


In [ ]:
import torch, time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Lectores disponibles en:', device)

def atencion(Q, K, V):
    d_k = Q.shape[-1] # tamaño del vector
    pesos = torch.softmax(Q @ K.T / d_k ** 0.5, dim=-1)   # quién mira a quién
    return pesos @ V, pesos                                # mezcla de información

# --- Parte 1: la frase «El gato duerme» (los números de la clase) ---
Q = torch.tensor([[1., 0], [1, 2], [0, 2]], device=device)
K = torch.tensor([[1., 0], [0, 2], [1, 1]], device=device)
V = K.clone()
salida, pesos = atencion(Q, K, V)
print('Atención (filas: El, gato, duerme):')
print(pesos.cpu().round(decimals=2))
print('Vectores con contexto:')
print(salida.cpu().round(decimals=2))

# --- Parte 2: la carrera con 4096 palabras ---
def carrera(dev, n=4096, d=64, veces=10):
    Q, K, V = (torch.randn(n, d, device=dev) for _ in range(3))
    atencion(Q, K, V)                                      # calentamiento
    if dev == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(veces):
        atencion(Q, K, V)
    if dev == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / veces

t_cpu = carrera('cpu')
print(f'CPU: {t_cpu * 1000:.1f} ms por atención')
if torch.cuda.is_available():
    t_gpu = carrera('cuda')
    print(f'GPU: {t_gpu * 1000:.1f} ms por atención')
    print(f'La GPU fue {t_cpu / t_gpu:.0f} veces más rápida')

Lectores disponibles en: cuda
Atención (filas: El, gato, duerme):
tensor([[0.4000, 0.2000, 0.4000],
        [0.0700, 0.6200, 0.3100],
        [0.0500, 0.7700, 0.1900]])
Vectores con contexto:
tensor([[0.8000, 0.8000],
        [0.3800, 1.5500],
        [0.2300, 1.7200]])
CPU: 292.8 ms por atención
GPU: 3.4 ms por atención
La GPU fue 86 veces más rápida


In [ ]:
import torch, math, time

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Las 4 matrices mágicas de los videojuegos (4x4) ---
def escalar(s):
    return torch.diag(torch.tensor([s, s, s, 1.0]))

def rotar_y(grados):
    a = math.radians(grados); c, s = math.cos(a), math.sin(a)
    return torch.tensor([[ c, 0, s, 0],
                         [ 0, 1, 0, 0],
                         [-s, 0, c, 0],
                         [ 0, 0, 0, 1]])

def trasladar(x, y, z):
    M = torch.eye(4)
    M[0, 3], M[1, 3], M[2, 3] = x, y, z
    return M

def proyectar(f=1.0):          # perspectiva: la cuarta fila copia z para dividir luego
    return torch.tensor([[f, 0, 0, 0], [0, f, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0]])

# --- Parte 1: un cubo (8 vértices) ---
cubo = torch.tensor([[x, y, z, 1.] for x in (-1, 1) for y in (-1, 1) for z in (-1, 1)]).T
M = proyectar() @ trasladar(0, 0, 5) @ rotar_y(45) @ escalar(1.0)   # todo en UNA matriz
P = M @ cubo                                  # transforma los 8 vértices a la vez
print('Cubo en pantalla (x, y):')
print((P[:2] / P[3]).T.round(decimals=2))     # dividir entre w da la perspectiva

# --- Parte 2: un personaje con 5 millones de vértices, 60 fotogramas ---
def render(dev, n=5_000_000, frames=60):
    V = torch.rand(4, n, device=dev); V[3] = 1.0          # n vértices (x, y, z, 1)
    if dev == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for f in range(frames):                                # cada fotograma gira 6 grados
        M = (proyectar() @ trasladar(0, 0, 5) @ rotar_y(f * 6) @ escalar(1.0)).to(dev)
        P = M @ V
        pantalla = P[:2] / P[3]
    if dev == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / frames * 1000     # ms por fotograma

for dev in ['cpu'] + (['cuda'] if torch.cuda.is_available() else []):
    ms = render(dev)
    print(f'{dev}: {ms:.1f} ms por fotograma ({1000 / ms:.0f} fps)')
print('Meta de un videojuego: 16,7 ms por fotograma (60 fps)')

Cubo en pantalla (x, y):
tensor([[-0.2800, -0.2000],
        [ 0.0000, -0.1600],
        [-0.2800,  0.2000],
        [ 0.0000,  0.1600],
        [ 0.0000, -0.2800],
        [ 0.2800, -0.2000],
        [ 0.0000,  0.2800],
        [ 0.2800,  0.2000]])
cpu: 75.3 ms por fotograma (13 fps)
cuda: 4.9 ms por fotograma (203 fps)
Meta de un videojuego: 16,7 ms por fotograma (60 fps)
